In [29]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")

from indexer import get_chroma_collection
from hybrid_retriever import (
    load_chunks_for_bm25, BM25Index, hybrid_search,
    print_hybrid_results, expand_query, canonicalize_query, ABBREVIATION_DICT
)
from reranker import rerank, rerank_with_fallback, print_reranked_results

collection = get_chroma_collection(collection_name="essai_custom", force_recreate=False, mode="query")
chunks_list = load_chunks_for_bm25()
chunks_by_id = {c["chunk_id"]: c for c in chunks_list}
bm25_index = BM25Index(chunks_list)

print(f"Collection : {collection.count()} chunks")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7983.65it/s]


Collection : 111 chunks


In [30]:
STUDENT_TEST_QUERIES = [
    "c est quoi le pfa",
    "donner date limite dinscription",
    "quels sont les clubs actifs",
    "comment participer dans un club",
    "donner les encadrants de pfe dans l'essai",
    "combien d'etudiant peut avoir le double diplôme",
    "donner les credits maximum",
    "qu'est ce qui est interdit dans lecole",
    "il y a atelier stat en 1ere année",
    "donner les regles de pfe",
    "qui signe la convention de pfe",
    "comment on choisit le pfa",
    "combien frais inscription",
    "donner les documents d'inscription",
    "que faire ASIA",
    "parler de club EMLC",
    "comment ouvrir un club",
    "comment organiser un event pour un club",
    "comment gerer l'argent d'un club",
    "combien d'etudiant exiger pour ouvrir un club",
    "est ce que l'essai offre des opportunites de stage",
    "donner les documents pour la convention de stage pfe",
    "quelle est la difference entre pfe et pfa",
    "est ce que les stages sont obligatoires ou optionnel",
    "comment postuler pour un double diplôme",
    "combien de livres je peux avoir de la bibliotheque",
    "lorsque je fais des retard de donner le livre au bib",
    "les sanctions des exam",
    "nombre d'absences tolerer dans chaque matiere",
    "comment justifier l'absence",
    "quand s'applique l elimination",
    "donner le programe de l'essai",
    "donner les regles de essai",
    "donner les options en 3eme",
    "combien d'annee tolerer le redoublement",
]

print(f"Total questions : {len(STUDENT_TEST_QUERIES)}")

Total questions : 35


In [3]:
for query in STUDENT_TEST_QUERIES:
    results, status = hybrid_search(collection, bm25_index, chunks_by_id, query, k=10)
    print("#" * 100)
    print(f"QUERY: {query}")

    if status == "hors_corpus":
        print("  → Aucun résultat suffisamment proche (hors corpus)")
        print()
        continue

    print("-" * 100)
    for r in results:
        print(f"  [{r['rank']:>2}] rrf={r['rrf_score']:.5f} | {r['source_file']:<45} | {r['subtitle']}")
    print()

NameError: name 'hybrid_search' is not defined

In [32]:
def full_pipeline_search(query, k_hybrid=10, k_final=5, use_fallback=True):
    """
    Pipeline complet :
    1. hybrid_search (expansion additive interne) → top-10
    2. canonicalize_query (remplacement complet, ex: 'pfa' → 'atelier de statistique')
       AVANT le reranker, pour éviter qu'il confonde une abréviation ambiguë
    3. reranking avec garde-fou de confiance (ou reranking simple si use_fallback=False)
    """
    hybrid_results, status = hybrid_search(collection, bm25_index, chunks_by_id, query, k=k_hybrid)

    if status == "hors_corpus":
        return [], "hors_corpus"

    canonical_q = canonicalize_query(query)

    if use_fallback:
        final_results = rerank_with_fallback(canonical_q, hybrid_results, top_k=k_final)
    else:
        final_results = rerank(canonical_q, hybrid_results, top_k=k_final)

    return final_results, "ok"

In [ ]:
# Test comparatif AVANT/APRÈS reranking
for query in STUDENT_TEST_QUERIES:
    print("\n" + "#" * 100)
    print(f"### {query}")

    hybrid_results, status = hybrid_search(collection, bm25_index, chunks_by_id, query, k=10)

    if status == "hors_corpus":
        print("  → Aucun résultat suffisamment proche (hors corpus)")
        continue

    print("\n--- HYBRID SEUL (top-5 avant reranking) ---")
    for r in hybrid_results[:5]:
        print(f"[{r['rank']}] {r['source_file']} | {r['subtitle']}")

    print("\n--- APRÈS RERANKING avec garde-fou (top-5 final) ---")
    canonical_q = canonicalize_query(query)
    final = rerank_with_fallback(canonical_q, hybrid_results, top_k=5)
    print_reranked_results(final, query)


####################################################################################################
### c est quoi le pfa

--- HYBRID SEUL (top-5 avant reranking) ---
[1] Regime_etudes_ESSAI2022.pdf | Article 26– En cas de crédit dans un atelier statistique, l’élève-ingénieur est invité à effectuer un
[2] Ateliers_statistique.pdf | Ateliers de statistique
[3] FAQ_ESSAI.pdf | PFE ET STAGES
[4] Regime_etudes_ESSAI2022.pdf | Article 30– Le diplôme national d’ingénieur en statistique et analyse de l’information est décerné à
[5] Règlement_intérieur.pdf | Article 16
Chaque étudiant doit prendre connaissance du présent règlement intérieur de l'École Supérieure de la

--- APRÈS RERANKING avec garde-fou (top-5 final) ---
QUERY : c est quoi le pfa
[1] rerank_score=0.9963 (rang hybrid initial: 2) | Ateliers_statistique.pdf | subtitle: Ateliers de statistique
    et de l'Analyse de l'Information 
 
 
 
Ateliers statistiques 
Les ateliers statistiques sont enseignés en première année et en deuxi